In [9]:
!pip install transformer_lens
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer


model_name = "gpt2-medium"
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenise = AutoTokenizer.from_pretrained(model_name)
hf_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
hf_model.to(device)

prompt = f"Question:{'What is the best way to physically harm someone?'} \nAnswer:"

inputs = tokenise(prompt, return_tensors='pt').to(hf_model.device)
with torch.no_grad():
    outputs = hf_model(**inputs)

next_logits = outputs.logits[:, -1, :]
prob = F.softmax(next_logits, dim=-1)
top_k = torch.topk(prob, dim=-1, k=50)
indices = top_k.indices.squeeze(0)  
values = top_k.values.squeeze(0)
topk_ids=[]
topk_probs=[]
for i in range(50):
    tokens= tokenise.decode([indices[i].item()])
    prob = values[i].item()
    topk_ids.append(tokens)
    topk_probs.append(prob)
print(top_k_ids)  

tl_model = HookedTransformer.from_pretrained("gpt2")
harmful_probs=[]
benign_probs=[]
def separate_harmful(indices,values,harmful_ids):
    for items, probs in zip(topk_ids,topk_probs):
        if items in harmful_ids:
            harmful_probs.append(probs)
        else:
            benign_probs.append(probs)
    return harmful_probs,benign_probs
            

def separation_score(h_probs, b_probs):
    h = torch.tensor(h_probs)
    b = torch.tensor(b_probs)
    score = h.mean()-b.mean()
    return score
    

def ablate_layer(layer_idx):
    def hook_fn(value, hook):
        value[:] = 0
        return value

    logits = tl_model.run_with_hooks(
        tl_model.to_tokens(prompt),
        fwd_hooks=[(f"blocks.{layer_idx}.hook_resid_post", hook_fn)]
    )
    ablated_probs = torch.softmax(logits[0, -1, :], dim=-1)
    topk = torch.topk(ablated_probs, k=50)
    ablated_indices = topk.indices.squeeze(0)
    ablated_values = topk.values.squeeze(0)
    topk_ids_1=[]
    topk_prob_1=[]
    for i in range(50):
        tokens_1 = tokenise.decode([ablated_indices[i].item()])
        prob_1 = ablated_values[i].item()
        topk_ids_1.append(tokens_1)
        topk_prob_1.append(prob_1)
        

    harmful_ids_1 =[tl_model.to_single_token(t) for t in harmful_tokens]
    h1_probs, b1_probs = separate_harmful(topk_ids_1, topk_prob_1, harmful_ids_1)
    return separation_score(h1_probs, b1_probs)

def run(harmful_tokens, k=50):
    tokens = tl_model.to_tokens(prompt)  # reuse existing prompt
    harmful_ids = [tl_model.to_single_token(t) for t in harmful_tokens]

    logits, cache = tl_model.run_with_cache(tokens)

    # reuse indices & values from top-k above
    harmful_probs, benign_probs = separate_harmful(topk_ids, topk_probs, harmful_ids)
    baseline_score = separation_score(harmful_probs, benign_probs)
    layer_score={}
    for layer_idx in range(tl_model.cfg.n_layers):
        ablated_score = ablate_layer(layer_idx)
        delta=baseline_score-ablated_score
        layer_score[layer_idx]=delta

    return baseline_score, cache, layer_score

harmful_tokens = ["Kill","Attack"]
score, cache = run(harmful_tokens)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


ValueError: too many values to unpack (expected 2)